## 1. Import Libraries and Setup

In [11]:
import os
import lancedb
import subprocess
import requests
import time
from pathlib import Path
from datasets import load_dataset
from dotenv import load_dotenv

# LlamaIndex Core Components
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Document
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser
from llama_index.core.ingestion import IngestionPipeline

# Embeddings and vector store
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.lancedb import LanceDBVectorStore

# LLM integrations
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.llms.ollama import Ollama

# Async support for notebooks
import nest_asyncio
nest_asyncio.apply()
load_dotenv()

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. Data Preparation and Loading

In [12]:
def prepare_data(num_samples = 100):
    """
    Load dataset and create document files
    """
    print(f"Loading {num_samples} personas from dataset...")
    
    # Load the personas dataset
    dataset = load_dataset("dvilasuero/finepersonas-v0.1-tiny", split="train")
    
    # Create data directory
    Path("data").mkdir(parents=True, exist_ok=True)
    
    # Save personas as text files and create Document objects
    documents = []
    for i, persona in enumerate(dataset.select(range(min(num_samples, len(dataset))))):
        # Create Document objects for LlamaIndex
        doc = Document(
            text = persona["persona"],  # type: ignore[index]
            metadata = {
                "persona_id": i,
                "source": "finepersonas-dataset"
            } 
        )
        documents.append(doc)
        
        # Optionally save to files as well
        with open(Path("data") / f"persona_{i}.txt", "w", encoding="utf-8") as f:
            f.write(persona["persona"])  # type: ignore[index]
            
    print(f"Prepared {len(documents)} documents")
    return documents

documents = prepare_data(num_samples=100)

Loading 100 personas from dataset...
Prepared 100 documents


## 3. LanceDB Vector Store Setup

In [13]:
def setup_lancedb_store(table_name="personas_rag"):
    """
    Initialize LanceDB and create/connect to a table
    """
    print("Setting up LanceDB connection...")
    
    # Create/connect to LanceDB
    db = lancedb.connect("./lancedb_data")
    
    # LlamaIndex will handle table creation with proper schema
    print(f"Connected to LanceDB, table: {table_name}")
    
    return db, table_name

db, table_name = setup_lancedb_store()

Setting up LanceDB connection...
Connected to LanceDB, table: personas_rag


## 4. Vector Embeddings and Ingestion Pipeline

In [14]:
async def create_and_populate_index(documents, db, table_name):
    """
    Create ingestion pipeline and populate LanceDB with embeddings. 
    """
    print("Creating Embedding Model and Ingestion Pipeline...")
    
    # Initialize embedding model
    embed_model = HuggingFaceEmbedding(
        model_name="BAAI/bge-small-en-v1.5"
    )
    
    # Create LanceDB vector store
    vector_store = LanceDBVectorStore(
        uri="./lancedb_data",
        table_name=table_name,
        mode="overwrite"    # overwrite existing table
    )
    
    # Create ingestion pipeline
    pipeline = IngestionPipeline(
        transformations=[
            SentenceSplitter(chunk_size=100, chunk_overlap=20),
            embed_model
        ],
        vector_store=vector_store
    )
    
    print("Processing documents and creating embeddings...")
    # Run the pipeline to process documents and store in LanceDB
    nodes = await pipeline.arun(documents = documents)
    print(f"Successfully processed {len(nodes)} text chunks")
    
    return vector_store, embed_model

vector_store, embed_model = await create_and_populate_index(documents, db, table_name)

Creating Embedding Model and Ingestion Pipeline...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8763.82it/s]


Processing documents and creating embeddings...
Successfully processed 100 text chunks


## 5. Option 1: Vector Search Only (No LLM)

In [15]:
def perform_vector_search(db, table_name, query_text, embed_model, top_k=5):
    """
    Perform direct vector search on LanceDB. 
    """
    # Get query embedding
    query_embedding = embed_model.get_text_embedding(query_text)
    
    # Open table & perform search
    table = db.open_table(table_name)
    results = table.search(query_embedding).limit(top_k).to_pandas()
    
    return results

def test_vector_search():
    """
    Test vector search functionality with sample queries. 
    """
    print("Testing vector search (no LLM needed)")
    print("="*50)
    
    # Test queries
    queries = [
        "Technology and Artificial Intelligence Expert",
        "Teacher educator professor",
        "Environment Climate Sustainability",
        "Art Culture Heritage Creative"
    ]
    
    for query in queries:
        print(f"\nQuery: {query}")
        print("-"*30)
        
        results = perform_vector_search(db, table_name, query, embed_model, top_k=3)
        
        for idx, row in results.iterrows():
            score = row.get("_distance", "N/A")
            text = row.get("text", "N/A")
            
            # Format score
            if isinstance(score, (int, float)):
                score_str = f"{score:.3f}"
            else:
                score_str = str(score)

            print(f"\nResult {idx + 1} (Score: {score_str}):")
            print(f"{text[:200]}...")
            
test_vector_search()

Testing vector search (no LLM needed)

Query: Technology and Artificial Intelligence Expert
------------------------------

Result 1 (Score: 0.589):
A computer scientist or electronics engineer researching alternative sustainable materials for neuromorphic computing and memristor development or A science journalist covering emerging technologies a...

Result 2 (Score: 0.589):
A computer scientist or electronics engineer researching alternative sustainable materials for neuromorphic computing and memristor development or A science journalist covering emerging technologies a...

Result 3 (Score: 0.589):
A computer scientist or electronics engineer researching alternative sustainable materials for neuromorphic computing and memristor development or A science journalist covering emerging technologies a...

Query: Teacher educator professor
------------------------------

Result 1 (Score: 0.577):
An English language arts teacher with a focus on upper elementary education....

Result 2 (Scor

## 6. Option 2: RAG with HuggingFace API 

In [ ]:
def create_query_engine(vector_store, embed_model, llm=None):
    """
    Create a query engine from the vector store. 
    """
    
    # Create index from vector store
    index = VectorStoreIndex.from_vector_store(
        vector_store=vector_store,
        embed_model=embed_model
    )
    
    # Setup LLM if provided
    query_engine_kwargs = {}
    if llm:
        query_engine_kwargs['llm'] = llm
    
    # Create query engine
    query_engine = index.as_query_engine(
        response_mode="tree_summarize",
        **query_engine_kwargs
    )
    
    return query_engine

def query_rag(query_engine, question):
    """
    Query the RAG system and return response. 
    """
    response = query_engine.query(question)
    return response

async def test_huggingface_rag():
    """
    Test RAG with HuggingFace API 
    """
    print("Testing RAG with HuggingFace API")
    print("=" * 40)
    
    try:
        llm = HuggingFaceInferenceAPI(
            model_name="meta-llama/Llama-3.1-8B-Instruct",
            provider="auto",
            token=os.getenv("HF_TOKEN")
        )
        
        # Create query engine
        query_engine = create_query_engine(vector_store, embed_model, llm)
        
        # Test queries
        queries = [
            "Find personas interested in technology and AI.",
            "Who are the educators or teachers in the data set?",
            "Describe personas working with environmental topics."
        ]
        
        for query in queries:
            print(f"\nQuery: {query}")
            print("-" * 30)
            
            try:
                response = query_rag(query_engine, query)
                print(f"Response: {response}")
            except Exception as e:
                print(f"Error: {e}")
                
    except Exception as e:
        print(f"Setup error: {e}")
        print("Make sure to set your HuggingFace API key above")
        
await test_huggingface_rag()

Testing RAG with HuggingFace API

Query: Find personas interested in technology and AI.
------------------------------
Error: (Request ID: Root=1-6a198968-252a910f42d890817d11a3f7;cc42d2dd-70eb-4d1d-bbbf-2876ee29f378)

Bad request:
{'message': "The requested model 'mistralai/Mistral-7B-Instruct-v0.3' is not a chat model.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}

Query: Who are the educators or teachers in the data set?
------------------------------
Error: (Request ID: Root=1-6a198968-7fecc22765583e8d0abcf013;4cf3600c-a91e-4e51-a89a-f01c0f57ac77)

Bad request:
{'message': "The requested model 'mistralai/Mistral-7B-Instruct-v0.3' is not a chat model.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}

Query: Describe personas working with environmental topics.
------------------------------
Error: (Request ID: Root=1-6a198968-35d19b7f76f0505d628ca029;163bdd6a-870c-4574-bab7-4ba56f623fca)

Bad request:
{'message'